In [6]:
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd
from pyproj import Proj
from itertools import combinations
from scipy.spatial.distance import euclidean
from scipy.stats import skew, kurtosis
import pprint



# Albers Equal Area projection (conus_albers projection)
proj_albers = Proj(proj='aea', lat_1=29.5, lat_2=45.5, lon_0=-96, lat_0=37.5, datum='WGS84')

# Loading the data
ska_state_data_path = '/Users/fardeenb/Documents/Projects/cs3891-network-project-test/data/raw_data/ska_state.csv'
df = pd.read_csv(ska_state_data_path, delimiter=',')


#Filter data by state levels
state_data = df[df['region_code'].str.len() == 2]
print(state_data)

            region region_code  period  all_providers  \
0          Alabama          AL    2011           9676   
1           Alaska          AK    2011           1933   
2          Arizona          AZ    2011          14530   
3         Arkansas          AR    2011           5456   
4       California          CA    2011          76529   
..             ...         ...     ...            ...   
145       Virginia          VA    2013          18421   
146     Washington          WA    2013          16857   
147  West Virginia          WV    2013           4641   
148      Wisconsin          WI    2013          13842   
149        Wyoming          WY    2013           1296   

     all_primary_care_providers  all_physicians  all_primary_care_physicians  \
0                          4064            8281                         3341   
1                           975            1445                          647   
2                          5898           11622                         431

In [7]:
# Cols of interest
columns_of_interest = [
    'all_providers', 
    'all_primary_care_providers', 
    'all_physicians', 
    'all_primary_care_physicians', 
    'all_nurse_practitioners', 
    'all_primary_care_nurse_practitioners', 
    'all_physician_assistants', 
    'all_primary_care_physician_assistants'
]





# Summary Statistics
means = {}
std_devs = {}
minimums = {}
q1s = {}
medians = {} # q2s 
q3s = {}
maximums = {}
variances = {}
ranges = {}
sum_of_squares = {}
coefficients_of_variation = {}
skewnesses = {}
kurtoses = {}


# Calculate summary statistics for each column
for column in columns_of_interest:
    data = df[column]

    
    # Central Tendency & Spread
    means[column] = round(float(data.mean()), 2)
    std_devs[column] = round(float(data.std()), 2)
    variances[column] = round(float(data.var()), 2)
    ranges[column] = round(float(data.max() - data.min()), 2)
    sum_of_squares[column] = round(float((data ** 2).sum()), 2)
    minimums[column] = round(float(df[column].min()), 2)
    maximums[column] = round(float(df[column].max()), 2)
    
    # IQR Q1, Q2 (median), Q3
    q1s[column] = round(float(df[column].quantile(0.25)), 2)
    medians[column] = round(float(df[column].median()), 2)
    q3s[column] = round(float(df[column].quantile(0.75)), 2)

    # Coefficient of Variation
    coefficients_of_variation[column] = round(float(std_devs[column] / means[column] * 100), 2)
    
    # Skewness and Kurtosis
    skewnesses[column] = round(float(skew(data)), 2)
    kurtoses[column] = round(float(kurtosis(data)), 2)
    
summary_statistics = {
    "Means": means,
    "Standard Deviations": std_devs,
    "Minimums": minimums,
    "Variances": variances,
    "25th Percentiles (Q1)": q1s,
    "Medians (50th Percentiles)": medians,
    "75th Percentiles (Q3)": q3s,
    "Maximums": maximums,
    "Ranges": ranges,
    "Sum of Squares": sum_of_squares,
    "Coefficients of Variation (%)": coefficients_of_variation,
    "Skewness": skewnesses,
    "Kurtosis": kurtoses
}


pprint.pprint(summary_statistics, width=80, sort_dicts=False)
 






{'Means': {'all_providers': 14530.0,
           'all_primary_care_providers': 6123.8,
           'all_physicians': 12023.36,
           'all_primary_care_physicians': 4746.81,
           'all_nurse_practitioners': 1465.49,
           'all_primary_care_nurse_practitioners': 902.13,
           'all_physician_assistants': 1041.63,
           'all_primary_care_physician_assistants': 474.87},
 'Standard Deviations': {'all_providers': 15174.63,
                         'all_primary_care_providers': 6243.96,
                         'all_physicians': 12986.23,
                         'all_primary_care_physicians': 5041.62,
                         'all_nurse_practitioners': 1254.03,
                         'all_primary_care_nurse_practitioners': 770.89,
                         'all_physician_assistants': 1084.16,
                         'all_primary_care_physician_assistants': 526.54},
 'Minimums': {'all_providers': 1263.0,
              'all_primary_care_providers': 612.0,
              

In [8]:
import plotly.graph_objects as go
import pandas as pd



# Create a DataFrame from the summary statistics
summary_df = pd.DataFrame(summary_statistics)

# Transpose the DataFrame to switch rows and columns
transposed_df = summary_df.transpose()

# Create the plotly table
fig = go.Figure(data=[go.Table(
    header=dict(
        values=["Statistics", *[col.replace("(", "\n(").replace(")", ")\n") for col in transposed_df.columns]],  # Wrap long header text
        fill_color='paleturquoise', 
        align='left',
        font=dict(size=12)  # Adjust font size for readability
    ),
    cells=dict(
        values=[list(transposed_df.index)] + [transposed_df[col].values for col in transposed_df.columns],
        fill_color='lavender',
        align='left',
        font=dict(size=10)  # Adjust font size for readability
    )
)])

# Update layout for a cleaner look
fig.update_layout(
    title="Summary Statistics for Different Provider Types",
    title_x=0.5,
    width=800,  # Set a fixed width
    height=600,  # Set a fixed height
    margin=dict(l=20, r=20, t=40, b=20)
)

# Show the table
fig.show()